# Module 08: Concurrency and Parallelism for Java Developers

## Interview Pro-Tip

This module covers THE most common Python interview question for backend engineers:
**"What is the GIL and how do you run CPU-bound tasks in parallel?"**

## Java Recall

```java
// Java threads ARE parallel (for CPU work)
ExecutorService executor = Executors.newFixedThreadPool(4);
List<Future<Integer>> futures = new ArrayList<>();

for (int i = 0; i < 4; i++) {
    futures.add(executor.submit(() -> heavyComputation()));
}

for (Future<Integer> f : futures) {
    total += f.get();
}
executor.shutdown();
```

In Java, threads give you real parallelism on multi-core CPUs. The JVM has no GIL.
For CPU-bound work, you use threads. For I/O-bound, you use async/CompletableFuture or threads.

## Python Way — The GIL

### What is the GIL?

The **Global Interpreter Lock** (GIL) is a mutex that prevents multiple threads from
executing Python bytecode simultaneously. Only ONE thread can hold the GIL at any moment.

### Why does it exist?

CPython's memory management uses reference counting (not a tracing GC like Java).
Every object has a refcount that gets incremented/decremented. Without a lock, two
threads modifying the same object's refcount would corrupt memory. The GIL is the
simplest way to make CPython thread-safe internally.

### What this means in practice:

```python
# Threading for CPU work — DOES NOT HELP (GIL!)
# Threading for I/O work — WORKS FINE (GIL released during I/O)
```

The GIL is released during I/O operations, C extension calls, and `time.sleep()`.

## Three Ways to Achieve Parallelism (Without Third-Party Libraries)

### 1. `multiprocessing` — Separate processes, separate GILs
```python
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor(max_workers=4) as executor:
    results = executor.map(cpu_heavy, data)  # Truly parallel!
```
Each process has its own Python interpreter and its own GIL.

### 2. `concurrent.futures` — Unified API for threads and processes
```python
# I/O-bound: ThreadPoolExecutor
with ThreadPoolExecutor() as pool:
    results = pool.map(fetch_url, urls)

# CPU-bound: ProcessPoolExecutor
with ProcessPoolExecutor() as pool:
    results = pool.map(compute, data)
```
Same API, different executor.

### 3. `asyncio` — Cooperative multitasking (single-threaded)
```python
import asyncio

async def fetch(url):
    # async I/O operations go here
    pass

async def main():
    tasks = [fetch(url) for url in urls]
    results = await asyncio.gather(*tasks)

asyncio.run(main())
```
Single-threaded, cooperative, event-loop based. Great for high-concurrency I/O.
NOT for CPU work. Think of it as Node.js's event loop, but with nicer syntax.

## Key Differences

| Concept | Java | Python |
|---------|------|--------|
| Threads are parallel? | Yes (JVM has no GIL) | No (GIL limits to one thread) |
| CPU-bound parallelism | Threads | `multiprocessing` / `ProcessPoolExecutor` |
| I/O concurrency | Threads or CompletableFuture | Threads or `asyncio` |
| Async model | CompletableFuture / virtual threads | `asyncio` (explicit async/await) |
| Data sharing | Shared memory (with synchronization) | Processes: pickle + IPC / Threads: shared objects (GIL) |
| Memory overhead | Low (threads share heap) | High (each process has own interpreter) |
| GIL future | N/A | Being removed (PEP 703, Python 3.13 free-threaded build) |

## Interview Answer Template

**Q: "How do you run parallel CPU tasks in Python without third-party libraries?"**

A: "Python has a GIL, so threading won't help for CPU-bound work. Instead, three options:

1. **`multiprocessing`** — spawn separate Python processes. Each has its own GIL.
2. **`concurrent.futures`** — ThreadPoolExecutor (I/O) and ProcessPoolExecutor (CPU) with same API.
3. **`asyncio`** — cooperative multitasking for high-concurrency I/O. Not for CPU work.

For simple cases, ProcessPoolExecutor.map() is the modern, cleanest approach.
Python 3.13+ has a free-threaded build (PEP 703) that removes the GIL experimentally."

## Your Practice

Open `initial/practice.py`. Implement:

1. **sequential_sum** — Baseline sum (trivial)
2. **threaded_sum** — Using `ThreadPoolExecutor`
3. **process_sum** — Using `ProcessPoolExecutor`
4. **gil_demonstration** — Compare sequential vs threaded timing
5. **countdown** — Simple function for module sanity check

Run: `python run_test.py initial` / `python run_test.py complete`.

## References

- [Global Interpreter Lock (Python Wiki)](https://wiki.python.org/moin/GlobalInterpreterLock)
- [PEP 703 — Making the GIL Optional](https://peps.python.org/pep-0703/)
- [concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html)
- [multiprocessing](https://docs.python.org/3/library/multiprocessing.html)
- [asyncio](https://docs.python.org/3/library/asyncio.html)
